In [220]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# #Importing Model Data
    
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=data['time']
# # parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=data['time'].values/(1e9 * 60); times=times.astype(float);
# Np_str='125e3'
# #Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,140+1))
# # parcel=parcel.isel(time=np.arange(0,140+1))
# res='1km'

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
true_time=data['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
#Restricts the timesteps of the data from timesteps0 to 140
res='1km';t_res='5min';Np_str='1e6'
job_array=False;index_adjust=0
ocean_fraction=0.25


# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [221]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [222]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [223]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=42
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 683346, end_job = 700012


In [224]:
#Indexing Array with JobArray
parcel=parcel.isel(xh=slice(start_job,end_job))
#(for 150_000_000 parcels use 500-1000 jobs)

In [225]:
# Reading Back Data Later
##############
def make_data_dict(var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][:,start_job:end_job] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][:,start_job:end_job].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [226]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['A_g', 'A_c', 'W', 'QCQI', 'Z', 'Y', 'X']
data_dict = make_data_dict(var_names,read_type)
A_g, A_c, W, QCQI, Z, Y, X = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory()

Top 10 objects with highest memory usage
{'out': '24.0 MB', 'save': '24.0 MB', 'A_g': '17.73 MB', 'A_c': '17.73 MB', 'Z': '17.73 MB', 'Y': '17.73 MB', 'X': '17.73 MB', 'W': '8.87 MB', 'QCQI': '8.87 MB', 'QV': '8.87 MB'}

0.16326 GB in use overall


In [227]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'VARS_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'VARS_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['QV','TH','TH_E','BUOYANCY','HMC']
data_dict = make_data_dict(var_names,read_type)
QV, TH, TH_E, BUOYANCY, HMC = (data_dict[k] for k in var_names)
check_memory()

Top 10 objects with highest memory usage
{'out': '24.0 MB', 'save': '24.0 MB', 'A_g': '17.73 MB', 'A_c': '17.73 MB', 'Z': '17.73 MB', 'Y': '17.73 MB', 'X': '17.73 MB', 'W': '8.87 MB', 'QCQI': '8.87 MB', 'QV': '8.87 MB'}

0.16326 GB in use overall


In [228]:
################################################################################

In [ ]:
########################
#READING BACK IN
def LoadFinalData(in_file):
    dict = {}
    with h5py.File(in_file, 'r') as f:
        for key in f.keys():
            dict[key] = f[key][:]
    return dict

def LoadAllCloudBase():
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
    in_file = dir2 + f"all_cloudbase_{res}_{t_res}_{Np_str}.pkl"
    with open(in_file, 'rb') as f:
        all_cloudbase = pickle.load(f)
    return(all_cloudbase)
min_all_cloudbase=np.nanmin(LoadAllCloudBase())
print(f"Minimum Cloudbase is: {min_all_cloudbase}\n")

dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
in_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}"
final_dict=LoadFinalData(in_file)


#DYNAMICALLY CREATING VARIABLES
for key, value in final_dict.items():
    globals()[key] = value

# #DYNAMICALLY PRINTING VARIABLE SIZES
# for key in final_dict:
#     print(f"{key} has {final_dict[key].shape[0]} parcels")

# PRINTING VARIABLE SIZES (ONE BY ONE)
print(f'ALL: {len(CL_ALL_out_arr)} CL parcels and {len(nonCL_ALL_out_arr)} nonCL parcels')
print(f'SHALLOW: {len(CL_SHALLOW_out_arr)} CL parcels and {len(nonCL_SHALLOW_out_arr)} nonCL parcels')
print(f'DEEP: {len(CL_DEEP_out_arr)} CL parcels and {len(nonCL_DEEP_out_arr)} nonCL parcels')
print('\n')
print(f'ALL: {len(SBZ_ALL_out_arr)} SBZ parcels and {len(nonSBZ_ALL_out_arr)} nonSBZ parcels')
print(f'SHALLOW: {len(SBZ_SHALLOW_out_arr)} SBZ parcels and {len(nonSBZ_SHALLOW_out_arr)} nonSBZ parcels')
print(f'DEEP: {len(SBZ_DEEP_out_arr)} SBZ parcels and {len(nonSBZ_DEEP_out_arr)} nonSBZ parcels')
print('\n')
print(f'ALL: {len(ColdPool_ALL_out_arr)} ColdPool parcels')
print(f'SHALLOW: {len(ColdPool_SHALLOW_out_arr)} ColdPool parcels')
print(f'DEEP: {len(ColdPool_DEEP_out_arr)} ColdPool parcels')


#APPLYING JOB ARRAY
if "job_array" in globals():
    print('APPLYING JOB ARRAY')
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    for name in [
        'ALL_out_arr', 'ALL_save_arr',
        'SHALLOW_out_arr', 'SHALLOW_save_arr',
        'DEEP_out_arr', 'DEEP_save_arr',
        'ALL_SBZ_out_arr', 'ALL_nonSBZ_out_arr',
        'SHALLOW_SBZ_out_arr', 'SHALLOW_nonSBZ_out_arr',
        'DEEP_SBZ_out_arr', 'DEEP_nonSBZ_out_arr',
        'ALL_ColdPool_out_arr', 'SHALLOW_ColdPool_out_arr', 'DEEP_ColdPool_out_arr'
    ]:
        globals()[name] = job_filter(globals()[name])

In [236]:
#HISTOGRAM BIN SETTINGS
####################################
def GetBinSettings(var_name):    

    pre_calculated=False
    pre_calculated=True

    if pre_calculated==False:
        bin_settings = {
            'w':        (W.min(), W.max(), 1000),
            'qv':       (QV.min(), QV.max(), 1000),
            'qcqi':     (QCQI.min(), QCQI.max(), 1000),
            'th':       (TH.min(), TH.max(), 5000),
            'th_e':     (TH_E.min(), TH.max(), 5000),
            'buoyancy': (BUOYANCY.min(), BUOYANCY.max(), 1000),
            'HMC':      (HMC.min(), HMC.max(), 1000),
        }
    elif pre_calculated==True:
        bin_settings = {
            'w': (-18.99606, 47.273865, 1000),
            'qv': (9.235839e-07, 0.022054985, 1000),
            'qcqi': (0.0, 0.0061959606, 1000),
            'th': (297.87912, 463.44125, 5000),
            'th_e': (324.8358, 463.43524, 5000),
            'buoyancy': (-0.78747416, 0.599328, 1000),
            'HMC': (-0.00031354488, 0.0001856628, 1000)
        }

    
    # Select bin range based on var_name
    if var_name is not None and var_name in bin_settings:
        bin_left, bin_right, num_bins = bin_settings[var_name]
    else:
        # fallback default
        bin_left, bin_right, num_bins = -50, 50, 1000
    return bin_left,bin_right,num_bins

In [237]:
#RUNNING
################################################################################

In [238]:
def averaged_profiles(profile): 
    out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
    out_var=np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T #divides the data column by the counter column
    return out_var
def averaged_profile_count(profile): 
    out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
    counts=out_var[:, 1]
    zlevels=out_var[:, 2]
    return counts,zlevels

In [239]:
def CL_tracked_profile(var_data,var_name,type):
    # if var_name=='th': #TESTING
    #     var_data=(var_data-np.mean(var_data))/np.std(var_data)
    if type=='all':
        out_arr=ALL_out_arr.copy()
        # after_array=ALL_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_out_arr.copy()
        # after_array=SHALLOW_out_after_array
    elif type=='deep':
        out_arr=DEEP_out_arr.copy()
        # after_array=DEEP_out_after_array

    
    zhs=data['zh'].values
    # profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    # profile_array[:,2]=zhs;

    [bin_left,bin_right,num_bins]=GetBinSettings(var_name) 
    # if var_name=='th': #TESTING
    #     bin_left=var_data.min();bin_right=var_data.max() 
    profile_array =np.zeros((len(zhs), num_bins)) #TESTING***
    bin_list=np.linspace(bin_left,bin_right,num_bins)
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY INDEX_ADJUST
        ys=Y[ts,p-index_adjust] #JOBARRAY INDEX_ADJUST
        xs=X[ts,p-index_adjust] #JOBARRAY INDEX_ADJUST
        
        vars=var_data[ts,p-index_adjust] #JOBARRAY INDEX_ADJUST
        # print(vars)
        which_bins = np.clip(np.searchsorted(bin_list, vars) - 1, 0, num_bins - 1)
        
        for t, z, bin_idx in zip(ts, zs, which_bins):
            np.add.at(profile_array[:, bin_idx], z, 1)
    return profile_array

In [240]:
variables = {
    'w': W,
    'qv': QV,
    'qcqi': QCQI,
    'th': TH,
    'th_e': TH_E,
    'buoyancy': BUOYANCY,
    'HMC': HMC
}

types = ['all', 'shallow', 'deep']

bin_arrays = {}

for t in types:
    print(t.upper())
    for var_name, var_data in variables.items():
        print(var_name)
        key = f"{t.upper()}_profile_array_{var_name}"
        bin_arrays[key] = CL_tracked_profile(var_data, var_name, type=t)

ALL
w
qv
qcqi
th
th_e
buoyancy
HMC
SHALLOW
w
qv
qcqi
th
th_e
buoyancy
HMC
DEEP
w
qv
qcqi
th
th_e
buoyancy
HMC


In [241]:
#SAVING
import pickle
dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
output_file = dir2 + f"Tracked_Histograms_{res}_{Np_str}_5min_{job_id}.pkl"

# Save the entire bin_arrays dictionary
with open(output_file, 'wb') as f:
    pickle.dump(bin_arrays, f)

In [243]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOB_ARRAYS IS RUNNING
# recombine=True

In [255]:
def get_arrays(job_id):
    #LOADING BACK IN
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
    input_file = dir2 + f"Tracked_Histograms_{res}_{Np_str}_5min_{job_id}.pkl"
    # Load the dictionary from the pickle file
    with open(input_file, 'rb') as f:
        bin_arrays = pickle.load(f)
    return bin_arrays

if recombine==True:     
    job_id = 1
    dictionary1 = get_arrays(job_id)
    num_jobs = 60
    for job_id in np.arange(2, num_jobs + 1):
        if np.mod(job_id, 10) == 0:
            print(f"job_id = {job_id}")
        # job_id=1 #TESTING
        dictionary2 = get_arrays(job_id)
        
        # Add all items in dictionary2 to dictionary1 for each category and variable
        for category in ['ALL', 'SHALLOW', 'DEEP']:  # Categories: ALL, SHALLOW, DEEP
            for key in variables.keys():  # Variables: w, qv, qcqi, th, etc.
                # Assuming both dictionaries have the same structure and we're adding arrays
                dictionary1[f"{category}_profile_array_{key}"] += dictionary2[f"{category}_profile_array_{key}"]
    
    # Store the recombined dictionary using pickle
    dir3 = dir + 'Project_Algorithms/Tracked_Profiles/'
    output_file = dir3 + f"Tracked_Histograms_{res}_{Np_str}_5min.pkl"
    with open(output_file, 'wb') as f:
        pickle.dump(dictionary1, f)

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60
